# Part 4 — GenAI Call-Centre Review
**CareLoop Clinics — No-Show Guard**

This notebook implements Tasks 4.1–4.4. Keep `followup_notes.csv`, `appointments_clean.csv`, and `test_risk_scores.csv` in the same folder as this notebook.

**Security:** the Gemini API key is requested at runtime. Never paste it into a notebook cell, GitHub, or chat. If a key was previously shared publicly or in chat, revoke it and create a new key before running.

**Model compatibility:** `gemini-3.8-flash` may reject `thinking_level="minimal"`. The documented fallback below omits only `thinking_config`; `USE_MINIMAL_THINKING=False` is the safe default for this model. Set it to `True` only if your selected Flash model explicitly supports that setting.

Run cells in order. A fresh complete run makes 14 logical API requests: 1 Task 4.1 + 9 nonblank Task 4.2 notes + 4 Task 4.3 comparisons. N004 is blank and must not call the API. Requests are paced, transient errors receive limited retries, and Task 4.2 rows are checkpointed in notes_analysed_checkpoint.csv so rerunning the batch cell does not repeat completed note calls. Delete that checkpoint only when intentionally starting Task 4.2 from scratch. No labels are hardcoded into generated results.

## Setup and API configuration

In [7]:
%pip -q install -U google-genai pydantic pandas

import json
import time
import re
import os
import random
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError, field_validator

# Secure Key Entry
API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception:
        API_KEY = input("Paste your NEW Gemini API key: ").strip()

if not API_KEY:
    raise ValueError("API key is empty. Securely set GEMINI_API_KEY in Colab Secrets or input at runtime.")

client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-3.5-flash" # Automatically fall back to a stable, highly-available standard flash model
USE_MINIMAL_THINKING = False
API_CALLS = 0
LOGICAL_CALLS = 0
LAST_RAW_RESPONSE = ""
MIN_SECONDS_BETWEEN_REQUESTS = 5
MAX_TRANSIENT_RETRIES = 4
_LAST_REQUEST_AT = 0.0

def paced_generate_content(**kwargs):
    """Space requests and retry transient overload/rate errors with exponential backoff & jitter."""
    global API_CALLS, _LAST_REQUEST_AT
    delays = [5, 10, 20, 40]

    for attempt in range(MAX_TRANSIENT_RETRIES + 1):
        wait_for = MIN_SECONDS_BETWEEN_REQUESTS - (time.monotonic() - _LAST_REQUEST_AT)
        if wait_for > 0:
            time.sleep(wait_for)

        API_CALLS += 1
        _LAST_REQUEST_AT = time.monotonic()
        try:
            return client.models.generate_content(**kwargs)
        except Exception as exc:
            message = str(exc).lower()
            is_transient = ("429" in message or "resource_exhausted" in message
                           or "503" in message or "unavailable" in message
                           or "timeout" in message or "deadline" in message)

            if not is_transient or attempt >= MAX_TRANSIENT_RETRIES:
                raise

            # Calculate exponential backoff step with jitter
            base_delay = delays[min(attempt, len(delays)-1)]
            jitter = random.uniform(0.5, 1.5)
            delay = base_delay * jitter
            print(f"[Attempt {attempt+1}/{MAX_TRANSIENT_RETRIES}] Transient error ({exc}). Retrying in {delay:.2f}s...")
            time.sleep(delay)

# Part 4 - Defined schemas and constants to guarantee structural compliance
ALLOWED = [
    "forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
    "illness_or_emergency", "work_or_schedule", "other"
]

class FollowupInsight(BaseModel):
    root_cause: str
    confidence: float = Field(..., ge=0.0, le=1.0)
    summary: str = Field(..., max_length=150)
    recommended_action: str

    @field_validator("root_cause")
    @classmethod
    def validate_root_cause(cls, v):
        if v not in ALLOWED:
            raise ValueError(f"root_cause must be one of {ALLOWED}")
        return v

OUTPUT_COLUMNS = [
    "note_id", "appointment_id", "root_cause", "confidence",
    "summary", "recommended_action", "validation_status", "raw_response"
]

SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "root_cause": {"type": "STRING"},
        "confidence": {"type": "NUMBER"},
        "summary": {"type": "STRING"},
        "recommended_action": {"type": "STRING"}
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"]
}

SYSTEM_INSTRUCTION = (
    "You are a careful clinic operations analyst reviewing patient follow-up notes. "
    "Use only evidence in the note. Do not infer unsupported facts, diagnose, or invent details. "
    "Return only JSON matching the response schema."
)

notes = pd.read_csv("followup_notes.csv")
assert len(notes) >= 10 and list(notes.columns) == ["note_id", "appointment_id", "note_text"]
print("Notes loaded:", len(notes))

Notes loaded: 30


## Task 4.1 — Structured Gemini call with a CRAFT prompt

In [8]:
# Use the recommended stable production model to handle spikes in demand
MODEL_NAME = "gemini-3.5-flash"
USE_MINIMAL_THINKING = False

def make_prompt(note_text, version="V2"):
    if version == "V1":
        return f"Classify this note: {note_text}"

    return f"""
Context: CareLoop Clinics reviews patient follow-up notes to understand missed appointments.
Role: You are a careful healthcare operations analyst.
Action:
1. Select exactly one root_cause from this allowed list: {", ".join(ALLOWED)}.
2. Map explicit forgotten visit / no reminder / no SMS to forgot_or_no_reminder.
3. Map travel distance, unavailable transport, bus strike, or no auto to distance_or_transport.
4. Map consultation fee or affordability to cost_or_fee.
5. Map fever, illness, or emergency to illness_or_emergency.
6. Map work meeting, shift change, or inability to take leave to work_or_schedule.
7. Use other only when the note does not support one of the categories above; do not guess.
8. Set confidence between 0 and 1. Write a factual summary of at most 25 words and one practical action.
Tone: Neutral, concise, non-judgmental. Do not diagnose or invent facts.
Format: Return only JSON with root_cause, confidence, summary, recommended_action.
Patient note: {note_text}
"""

def analyse_note(note_text, version="V2"):
    """Return a dictionary containing exactly four required fields; raw JSON is stored separately."""
    global API_CALLS, LAST_RAW_RESPONSE, LOGICAL_CALLS

    config_args = dict(
        system_instruction=SYSTEM_INSTRUCTION,
        temperature=0,
        max_output_tokens=1000,
        response_mime_type="application/json",
        response_schema=SCHEMA
    )
    if USE_MINIMAL_THINKING:
        config_args["thinking_config"] = types.ThinkingConfig(thinking_level="minimal")

    config = types.GenerateContentConfig(**config_args)
    response = paced_generate_content(
        model=MODEL_NAME,
        contents=make_prompt(note_text, version),
        config=config
    )
    LOGICAL_CALLS += 1
    LAST_RAW_RESPONSE = response.text or ""
    parsed = json.loads(LAST_RAW_RESPONSE)

    required = {"root_cause", "confidence", "summary", "recommended_action"}
    if not isinstance(parsed, dict) or set(parsed.keys()) != required:
        raise ValueError(f"Gemini response must contain exactly {sorted(required)}; received: {parsed}")
    return parsed

# Test with the real N001 source row (not a synthetic example).
n001 = notes.loc[notes["note_id"].eq("N001")].iloc[0]
assert pd.notna(n001["note_text"]) and str(n001["note_text"]).strip()
n001_result = analyse_note(str(n001["note_text"]))
assert set(n001_result) == {"root_cause", "confidence", "summary", "recommended_action"}
print("N001 parsed dictionary:", n001_result)
print("N001 expected root_cause: forgot_or_no_reminder")
print("Task 4.1 API calls:", API_CALLS)

N001 parsed dictionary: {'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'Patient forgot the appointment because she received no reminder message and requested to rebook for next week.', 'recommended_action': 'Verify the automated reminder system and assist the patient with rebooking for next week.'}
N001 expected root_cause: forgot_or_no_reminder
Task 4.1 API calls: 1


## Task 4.2 — Batch processing and Pydantic validation

In [10]:
CHECKPOINT_PATH = "notes_analysed_checkpoint.csv"
OUTPUT_PATH = "notes_analysed.csv"

# Resume previously completed rows only when the checkpoint belongs to this exact input set.
completed = {}
if os.path.exists(CHECKPOINT_PATH):
    previous = pd.read_csv(CHECKPOINT_PATH, keep_default_na=False)
    if set(OUTPUT_COLUMNS).issubset(previous.columns):
        completed = {str(r["note_id"]): r.to_dict() for _, r in previous.iterrows()}
        print(f"Resuming from checkpoint: {len(completed)} saved rows")

rows = []
batch_api_calls_before = API_CALLS

for _, row in notes.head(10).iterrows():
    note_id = str(row["note_id"])
    note_text = row["note_text"]

    if note_id in completed:
        rows.append({col: completed[note_id].get(col, "") for col in OUTPUT_COLUMNS})
        print(f"Using checkpoint for {note_id}; no API request.")
        continue

    is_blank = pd.isna(note_text) or str(note_text).strip() == ""
    if is_blank:
        result_row = {
            "note_id": row["note_id"], "appointment_id": row["appointment_id"],
            "root_cause": "other", "confidence": 0.0,
            "summary": "No note recorded", "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank", "raw_response": ""
        }
        rows.append(result_row)
        pd.DataFrame(rows, columns=OUTPUT_COLUMNS).to_csv(CHECKPOINT_PATH, index=False)
        continue

    try:
        data = analyse_note(str(note_text))
        raw = LAST_RAW_RESPONSE
        validated = FollowupInsight(**data)
        result_row = {
            "note_id": row["note_id"], "appointment_id": row["appointment_id"],
            **validated.model_dump(), "validation_status": "valid", "raw_response": raw
        }
    except (ValidationError, ValueError, json.JSONDecodeError) as exc:
        raw = LAST_RAW_RESPONSE
        data = {}
        try:
            data = json.loads(raw) if raw else {}
        except json.JSONDecodeError:
            pass
        result_row = {
            "note_id": row["note_id"], "appointment_id": row["appointment_id"],
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", ""),
            "summary": data.get("summary", ""),
            "recommended_action": data.get("recommended_action", ""),
            "validation_status": "invalid", "raw_response": raw
        }
        print(f"Validation failure retained for {note_id}: {exc}")
    except Exception as exc:
        # Preserve completed rows and stop cleanly on API/quota errors.
        pd.DataFrame(rows, columns=OUTPUT_COLUMNS).to_csv(CHECKPOINT_PATH, index=False)
        print(f"Stopped at {note_id} due to API error: {exc}")
        print(f"Completed rows saved in {CHECKPOINT_PATH}. Fix/wait for quota, then rerun this cell.")
        raise

    rows.append(result_row)
    pd.DataFrame(rows, columns=OUTPUT_COLUMNS).to_csv(CHECKPOINT_PATH, index=False)

analysed = pd.DataFrame(rows, columns=OUTPUT_COLUMNS)
assert len(analysed) == 10
assert analysed.columns.tolist() == OUTPUT_COLUMNS
assert analysed.loc[analysed.note_id.eq("N004"), "validation_status"].iat[0] == "skipped_blank"
assert analysed.loc[analysed.note_id.eq("N004"), "raw_response"].iat[0] == ""
assert (analysed.loc[analysed.note_id.ne("N004"), "validation_status"].isin(["valid", "invalid"])).all()
valid_or_skipped = analysed.validation_status.isin(["valid", "skipped_blank"])
assert analysed.loc[valid_or_skipped, "confidence"].astype(float).between(0, 1).all()

analysed.to_csv(OUTPUT_PATH, index=False)
print("Task 4.2 API attempts this run:", API_CALLS - batch_api_calls_before)
print("Saved notes_analysed.csv:", analysed.shape)

print(analysed[["note_id", "root_cause", "confidence", "validation_status"]].to_string(index=False))
api_calls_made = API_CALLS - batch_api_calls_before
print("Task 4.2 API calls (must be 9 for a fresh run, or 0 when resuming):", api_calls_made)
assert api_calls_made in [0, 9], f"Unexpected API calls made: {api_calls_made}"
print("Saved notes_analysed.csv:", analysed.shape)

Resuming from checkpoint: 10 saved rows
Using checkpoint for N001; no API request.
Using checkpoint for N002; no API request.
Using checkpoint for N003; no API request.
Using checkpoint for N004; no API request.
Using checkpoint for N005; no API request.
Using checkpoint for N006; no API request.
Using checkpoint for N007; no API request.
Using checkpoint for N008; no API request.
Using checkpoint for N009; no API request.
Using checkpoint for N010; no API request.
Task 4.2 API attempts this run: 0
Saved notes_analysed.csv: (10, 8)
note_id            root_cause  confidence validation_status
   N001 forgot_or_no_reminder         1.0             valid
   N002 distance_or_transport         1.0             valid
   N003           cost_or_fee         1.0             valid
   N004                 other         0.0     skipped_blank
   N005  illness_or_emergency         1.0             valid
   N006      work_or_schedule         1.0             valid
   N007 forgot_or_no_reminder         1.0 

## Task 4.3 — Red-teaming and V1/V2 prompt comparison

In [14]:
# (a) Both intentionally malformed records must raise and be caught.
bad_cases = {
    "confidence_above_upper_bound": {
        "root_cause": "other", "confidence": 1.7,
        "summary": "Ambiguous patient note", "recommended_action": "Call the patient"
    },
    "missing_required_field": {
        "root_cause": "other", "confidence": 0.5,
        "summary": "Ambiguous patient note"
        # recommended_action intentionally missing
    }
}

red_team_results = {}
for case_name, bad_record in bad_cases.items():
    try:
        FollowupInsight(**bad_record)
        raise AssertionError(f"{case_name} unexpectedly passed validation")
    except ValidationError as exc:
        message = str(exc)
        red_team_results[case_name] = message
        print(f"CAUGHT {case_name}:")
        print(message)
        if case_name == "confidence_above_upper_bound":
            assert "less than or equal to 1" in message
        else:
            assert "missing" in message.lower() or "field required" in message.lower()

# (b) Compare V1 and V2 on the actual Hinglish N007 and ambiguous N009 notes.
comparison_rows = []
comparison_calls_before = LOGICAL_CALLS

for note_id in ["N007", "N009"]:
    source_row = notes.loc[notes["note_id"].eq(note_id)].iloc[0]
    note_text = str(source_row["note_text"])
    for version in ["V1", "V2"]:
        result = analyse_note(note_text, version=version)
        comparison_rows.append({
            "note_id": note_id,
            "prompt_version": version,
            "root_cause": result.get("root_cause", ""),
            "confidence": result.get("confidence", 0.0),
            "summary": result.get("summary", ""),
            "recommended_action": result.get("recommended_action", ""),
            "raw_response": LAST_RAW_RESPONSE
        })

comparison_df = pd.DataFrame(comparison_rows)
print("\n--- PROMPT COMPARISON RESULTS ---")
print(comparison_df[["note_id", "prompt_version", "root_cause", "confidence"]].to_string(index=False))

# Assert exactly 4 logical API calls were made in this step
assert LOGICAL_CALLS - comparison_calls_before == 4

# Assert that the total number of logical calls matches any valid run configuration
print("Total LOGICAL_CALLS so far:", LOGICAL_CALLS)
assert LOGICAL_CALLS >= 5, f"Unexpected total logical calls: {LOGICAL_CALLS}"
print("Total logical calls:", LOGICAL_CALLS)
print("Total actual HTTP attempts:", API_CALLS)

assert 12 <= LOGICAL_CALLS <= 14
assert API_CALLS < 25

print("API CALL ACCEPTANCE PASSED")

CAUGHT confidence_above_upper_bound:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
CAUGHT missing_required_field:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...Ambiguous patient note'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

--- PROMPT COMPARISON RESULTS ---
note_id prompt_version                                                           root_cause  confidence
   N007             V1 Patient forgot the appointment because no SMS reminder was received.        0.95
   N007             V2                                                forgot_or_no_reminder        0.98
   N009             V1                                     Patient's phone was switched off        1

## Task 4.4 — Merge, cross-tab, acceptance checks, and README

In [18]:
# Clearly marked field where you can insert your actual public GitHub URL before final run:
GITHUB_REPO_URL = "https://github.com/Aman8959/CareLoop-Clinics-No-Show-Guard.git"

clean = pd.read_csv("appointments_clean.csv")
risk = pd.read_csv("test_risk_scores.csv")

merged = analysed.merge(
    clean[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left",
    validate="one_to_one"
)
assert len(merged) == 10
assert merged["reminder_sent"].notna().all()
print("Merged shape:", merged.shape)
cross_tab = pd.crosstab(merged["root_cause"], merged["reminder_sent"])
print(cross_tab)

# Agreement with the 9 nonblank expected labels. N004 is excluded from agreement.
EXPECTED = {
    "N001": "forgot_or_no_reminder",
    "N002": "distance_or_transport",
    "N003": "cost_or_fee",
    "N005": "illness_or_emergency",
    "N006": "work_or_schedule",
    "N007": "forgot_or_no_reminder",
    "N008": "distance_or_transport",
    "N009": "other",
    "N010": "work_or_schedule"
}
actual = analysed.set_index("note_id")["root_cause"].to_dict()
agreement = sum(
    analysed.loc[analysed.note_id.eq(note_id), "validation_status"].iat[0] == "valid"
    and actual.get(note_id) == expected
    for note_id, expected in EXPECTED.items()
)
print(f"Expected-label agreement: {agreement}/9")
assert agreement >= 7, (
    "Acceptance requires at least 7/9 nonblank notes to agree. "
    "Review the actual raw responses and rerun only after resolving API/model issues; do not hardcode labels."
)

# Required reminder checks when the key expected classifications are matched.
key_notes = analysed[analysed.note_id.isin(["N001", "N007"])]
if set(key_notes.root_cause) == {"forgot_or_no_reminder"}:
    key_join = key_notes.merge(clean[["appointment_id", "reminder_sent"]], on="appointment_id")
    assert (key_join.reminder_sent == "No").sum() == 2
    print("Forgot/no-reminder key check: 2/2 have reminder_sent=No")

no_reminder_count = int((merged.reminder_sent == "No").sum())
print(f"Notes linked to appointments with reminder_sent=No: {no_reminder_count}/10")
assert no_reminder_count == 5

assert len(analysed) == 10 and len(analysed.columns) == 8
assert len(merged) == 10
assert int(clean.no_show.sum()) == 218
assert len(clean) == 990 and len(risk) == 198
assert {"appointment_id", "no_show", "risk_prob", "flag"}.issubset(risk.columns)

no_show_count = int(clean.no_show.sum())
no_show_pct = round(100 * clean.no_show.mean(), 1)
flagged_count = int(risk.flag.sum())
overall_cv_auc = 0.774  # From the verified Part 3 training CV output.
test_auc = 0.789        # From the verified Part 3 held-out test output.
clinic_count = 8
gomti_appointments = 90
gomti_no_show_pct = 33.3
lead_over_14 = 296
lead_over_14_pct = 29.9

readme = f"""# CareLoop Clinics — No-Show Guard

> **GitHub repository:** {GITHUB_REPO_URL}

## Project summary
- **Part 1 — Data preparation:** cleaned {len(clean):,} appointments; retained {no_show_count:,} no-shows ({no_show_pct:.1f}%).
- **Part 2 — SQL analysis:** analysed {clinic_count} clinics; Gomti Nagar recorded {gomti_no_show_pct:.1f}% no-shows across {gomti_appointments} appointments.
- **Part 3 — Prediction:** Logistic Regression achieved training 5-fold CV ROC-AUC ≈ {overall_cv_auc:.3f} and held-out test ROC-AUC ≈ {test_auc:.3f}; operating threshold = 0.35.
- **Part 4 — GenAI review:** analysed {len(analysed)} notes; {agreement}/9 nonblank classifications agreed with the expected categories; {no_reminder_count}/10 linked appointments had no reminder.

## Executive recommendation
CareLoop should prioritize appointments with long booking lead times and a history of missed visits for proactive confirmation: {lead_over_14} of {len(clean)} appointments ({lead_over_14_pct:.1f}%) were booked more than 14 days ahead, and the 31+ day lead-time segment had a 60.5% no-show rate in the SQL analysis. The overall dataset contained {no_show_count} no-shows out of {len(clean)} appointments ({no_show_pct:.1f}%), while Gomti Nagar had a {gomti_no_show_pct:.1f}% rate across {gomti_appointments} appointments. The Logistic Regression model is the preferred first-pass ranking model (CV ROC-AUC {overall_cv_auc:.3f}; test ROC-AUC {test_auc:.3f}); at threshold 0.35 it flags {flagged_count} of {len(risk)} held-out appointments for outreach. The call-centre should first confirm high-risk, long-lead appointments, offer rescheduling, and use the follow-up themes to address reminder delivery, transport access, affordability, illness, and work conflicts. GenAI classifications should support—not replace—staff review.

## Model choice and operating threshold
Logistic Regression was selected because it had the strongest 5-fold training CV ROC-AUC among the compared models (approximately {overall_cv_auc:.3f}) and achieved held-out ROC-AUC of approximately {test_auc:.3f}. The 0.35 threshold was selected using training-fold out-of-fold predictions to improve recall and F1; the held-out test set was evaluated once. Clinics should calibrate the operational threshold against call-centre capacity and the cost of empty appointment slots.

## Part 4 — Note review
- `notes_analysed.csv` contains 10 rows and 8 columns; N004 is recorded as `skipped_blank` with no API request.
- Expected-label agreement: {agreement}/9 nonblank notes.
- Linked appointments with `reminder_sent=No`: {no_reminder_count}/10.
- Actual Gemini API requests across Tasks 4.1–4.3: {API_CALLS}.

## Execution order
1. Run `01_ingest_clean.ipynb` to generate the cleaned data files.
2. Run the SQL schema/load/query scripts using the database engine specified in the project instructions.
3. Run `03_model.ipynb` to generate `test_risk_scores.csv`.
4. Run this notebook with a valid Gemini API key to generate `notes_analysed.csv` and this README.

## Limitations
1. Imputation and IQR outlier capping were performed before the train/test split, which can introduce preprocessing leakage.
2. The random row split may place appointments from the same patient in both train and test sets, making generalization to unseen patients look better than it is.
3. The dataset is observational; reminder associations and model feature effects do not establish causality.
4. The GenAI review uses only 10 notes, so its themes are illustrative and require human verification before operational use.

## Repository publication
Replace the placeholder repository URL at the top with the public GitHub repository URL. Confirm repository visibility is **Public** before submission.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)
print("README.md generated.")
print("Part 4 structural checks passed.")

Merged shape: (10, 9)
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1
Expected-label agreement: 9/9
Forgot/no-reminder key check: 2/2 have reminder_sent=No
Notes linked to appointments with reminder_sent=No: 5/10
README.md generated.
Part 4 structural checks passed.
